In [1]:
!pip install pandas numpy scipy statsmodels

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels


In [1]:
#Check whether the tail contraction is effective
import pandas as pd
from scipy.stats.mstats import winsorize

df_base = pd.read_csv("data.csv")
df_base.columns = df_base.columns.str.lower()
df_base["mthcaldt"] = pd.to_datetime(df_base["mthcaldt"])
df_base["ym"] = df_base["mthcaldt"].dt.to_period("M")

before = df_base["mthret"].copy()
after = df_base.groupby("ym")["mthret"].transform(lambda x: winsorize(x, limits=(0.001, 0.001)))
print("changed values:", (before != after).sum())

changed values: 5843


In [2]:
import numpy as np
import pandas as pd
from scipy.stats.mstats import winsorize

# Global parameters
J_LIST = [3, 6]
K_LIST = [3, 6]
SKIP = 1
PRICE_MIN = 5.0
DECILE = 0.10
MICROCAP_PCTL = 0.20  # NYSE 20th percentile market cap breakpoint
WINSOR_LIMITS = (0.001, 0.001)  # 0.1% / 99.9% cross-sectional winsorization

# ============================================================
# STEP 1: Load Data & Clean Duplicates
# ============================================================
df = pd.read_csv("data.csv")
df.columns = df.columns.str.lower()
df["mthcaldt"] = pd.to_datetime(df["mthcaldt"])
df["ym"] = df["mthcaldt"].dt.to_period("M")

initial_len = len(df)

# 1. Drop exact duplicate rows
df = df.drop_duplicates()
exact_dups = initial_len - len(df)

# 2. Detect key conflicts [permno, ym]
conflict_mask = df.duplicated(subset=["permno", "ym"], keep=False)
conflict_count = conflict_mask.sum()

# 3. Drop all conflicting rows
df = df[~conflict_mask].reset_index(drop=True)

print(f"Initial rows: {initial_len}")
print(f"Exact duplicates removed: {exact_dups}")
print(f"Partial duplicate conflicts removed: {conflict_count}")
print(f"Final rows remaining: {len(df)}")
print("-" * 50)

# ============================================================
# STEP 2: Reindex to Complete Monthly Calendar (Fix Gap Months)
# ============================================================
def reindex_stock(g):
    permno_val = g.name
    g = g.set_index("ym").sort_index()
    full_months = pd.period_range(g.index.min(), g.index.max(), freq="M")
    g = g.reindex(full_months)
    g.index.name = "ym"

    g["permno"] = permno_val
    g["sharetype"] = g["sharetype"].ffill().bfill()
    g["primaryexch"] = g["primaryexch"].ffill().bfill()
    return g.reset_index()

df = df.groupby("permno", group_keys=False).apply(reindex_stock).reset_index(drop=True)

# ============================================================
# STEP 3: Eligibility Flag
# ============================================================
df["eligible"] = (
    (df["mthprc"].abs() >= PRICE_MIN) &
    (df["sharetype"] == "NS") &
    (df["primaryexch"].isin(["N", "A", "Q"]))
)

nyse_breakpoint = (
    df[(df["primaryexch"] == "N") & df["mthcap"].notna()]
    .groupby("ym")["mthcap"]
    .quantile(MICROCAP_PCTL)
    .rename("mktcap_p20")
)
df = df.merge(nyse_breakpoint, on="ym", how="left")
df["eligible"] = df["eligible"] & (df["mthcap"] >= df["mktcap_p20"])

# ============================================================
# STEP 3.5: Winsorize returns cross-sectionally, by month
#   Toggle: set APPLY_WINSORIZE = True/False below to switch between
#   the baseline (no winsorize) and the robustness-check version.
#   Run the whole script once with each setting and compare results_df.
# ============================================================
APPLY_WINSORIZE = False  # <- baseline: False. Robustness check run: True.

if APPLY_WINSORIZE:
    df["mthret"] = df.groupby("ym")["mthret"].transform(
        lambda x: winsorize(x, limits=WINSOR_LIMITS)
    )

# ============================================================
# STEP 4: Formation Period Return R_i,t(J)
# ============================================================
def compute_formation_returns(data, J):
    d = data.sort_values(["permno", "ym"]).copy()

    # check how many returns needed clipping before computing log returns --
    # a large count would suggest a data quality issue upstream, not just
    # a few genuine extreme observations
    n_clipped = (d["mthret"] < -0.9999).sum()
    print(f"[J={J}] returns clipped at -0.9999: {n_clipped}")

    clean_ret = d["mthret"].clip(lower=-0.9999)
    d["log_ret"] = np.log1p(clean_ret)

    d[f"R_J{J}"] = (
        d.groupby("permno")["log_ret"]
        .transform(lambda x: np.expm1(x.shift(SKIP + 1).rolling(J, min_periods=J).sum()))
    )
    return d

# ============================================================
# STEP 5: Rank into Deciles (Winners / Losers)
# ============================================================
def rank_winners_losers(data, J):
    col = f"R_J{J}"
    d = data.dropna(subset=[col]).copy()
    d = d[d["eligible"]].copy()

    d["lo"] = d.groupby("ym")[col].transform(lambda x: x.quantile(DECILE))
    d["hi"] = d.groupby("ym")[col].transform(lambda x: x.quantile(1 - DECILE))

    d["group"] = np.where(
        d[col] <= d["lo"], "Loser",
        np.where(d[col] >= d["hi"], "Winner", None)
    )

    return d[d["group"].isin(["Winner", "Loser"])].copy()

# ============================================================
# STEP 6: Build K-Month Cohort Contributions
# ============================================================
def build_cohort_contributions(ranked, raw, K):
    raw_ret = raw[["permno", "ym", "mthret"]].rename(columns={"mthret": "ret_t"})

    rows = []
    for s, g in ranked.groupby("ym"):
        win = g[g["group"] == "Winner"].dropna(subset=["mthcap"])
        los = g[g["group"] == "Loser"].dropna(subset=["mthcap"])
        if len(win) == 0 or len(los) == 0:
            continue

        w_win_initial = win.set_index("permno")["mthcap"]
        w_los_initial = los.set_index("permno")["mthcap"]

        for k in range(1, K + 1):
            t = s + k
            ret_t = raw_ret[raw_ret["ym"] == t].set_index("permno")["ret_t"]

            win_ret = ret_t.reindex(win["permno"]).dropna()
            los_ret = ret_t.reindex(los["permno"]).dropna()

            if len(win_ret) == 0 or len(los_ret) == 0:
                continue

            umd_ew = win_ret.mean() - los_ret.mean()

            w_win_t = w_win_initial.reindex(win_ret.index)
            w_win_t = w_win_t / w_win_t.sum()

            w_los_t = w_los_initial.reindex(los_ret.index)
            w_los_t = w_los_t / w_los_t.sum()

            umd_vw = (w_win_t * win_ret).sum() - (w_los_t * los_ret).sum()

            rows.append({"hold_month": t, "UMD_EW": umd_ew, "UMD_VW": umd_vw})

    return pd.DataFrame(rows)

# ============================================================
# STEP 7: Overlapping Portfolio Returns & Performance
# ============================================================
def overlapping_portfolio(contrib_df, col):
    return contrib_df.groupby("hold_month")[col].mean().sort_index()

def performance(returns):
    r = returns.dropna()
    n = len(r)
    if n == 0:
        return {"n_obs": 0, "mean_monthly": np.nan, "cum_return": np.nan, "t_stat": np.nan, "sharpe": np.nan}

    mu = r.mean()
    sigma = r.std(ddof=1)
    se = sigma / np.sqrt(n) if n > 0 else np.nan
    cr = np.prod(1 + r) - 1
    t_stat = mu / se if se and se != 0 else np.nan
    sharpe = (mu / sigma) * np.sqrt(12) if sigma and sigma != 0 else np.nan

    return {
        "n_obs": n,
        "mean_monthly": mu,
        "cum_return": cr,
        "t_stat": t_stat,
        "sharpe": sharpe
    }

# ============================================================
# MAIN EXECUTION
# ============================================================
results = []
monthly_series_all = []
for J in J_LIST:
    df_J = compute_formation_returns(df, J)
    ranked = rank_winners_losers(df_J, J)

    for K in K_LIST:
        contrib = build_cohort_contributions(ranked, df, K)

        for col in ["UMD_EW", "UMD_VW"]:
            port = overlapping_portfolio(contrib, col)
            metrics = performance(port)
            metrics.update({"J": J, "K": K, "weight": col.replace("UMD_", "")})
            results.append(metrics)

            # save the monthly return series itself -- needed for the
            # five-factor regression later (summary stats alone aren't enough)
            monthly_df = port.reset_index()
            monthly_df.columns = ["hold_month", "wml_return"]
            monthly_df["J"] = J
            monthly_df["K"] = K
            monthly_df["weight"] = col.replace("UMD_", "")
            monthly_series_all.append(monthly_df)

results_df = pd.DataFrame(results)[
    ["J", "K", "weight", "n_obs", "mean_monthly", "cum_return", "t_stat", "sharpe"]
]

print("\n--- Final Backtest Results ---")
print(results_df)

output_name = "wml_results_winsorized.csv" if APPLY_WINSORIZE else "wml_results_baseline.csv"
results_df.to_csv(output_name, index=False)
print(f"Saved to: {output_name}")

# save the monthly WML return series -- this is what you need for the
# five-factor regression
monthly_series_df = pd.concat(monthly_series_all, ignore_index=True)
monthly_name = "wml_monthly_winsorized.csv" if APPLY_WINSORIZE else "wml_monthly_baseline.csv"
monthly_series_df.to_csv(monthly_name, index=False)
print(f"Saved monthly series to: {monthly_name}")

Initial rows: 1048575
Exact duplicates removed: 13089
Partial duplicate conflicts removed: 1066
Final rows remaining: 1034420
--------------------------------------------------
[J=3] returns clipped at -0.9999: 2
[J=6] returns clipped at -0.9999: 2

--- Final Backtest Results ---
   J  K weight  n_obs  mean_monthly  cum_return    t_stat    sharpe
0  3  3     EW    128      0.006899    1.230765  2.217329  0.678916
1  3  3     VW    128      0.008341    1.513466  1.982041  0.606874
2  3  6     EW    128      0.005149    0.816454  1.887242  0.577847
3  3  6     VW    128      0.005233    0.734553  1.375867  0.421271
4  6  3     EW    125      0.005797    0.822210  1.461650  0.452876
5  6  3     VW    125      0.008156    1.150321  1.444262  0.447488
6  6  6     EW    125      0.003606    0.400228  0.953362  0.295389
7  6  6     VW    125      0.004360    0.383746  0.831381  0.257594
Saved to: wml_results_baseline.csv
Saved monthly series to: wml_monthly_baseline.csv


In [3]:
import numpy as np
import pandas as pd
from scipy.stats.mstats import winsorize

# Global parameters
J_LIST = [3, 6]
K_LIST = [3, 6]
SKIP = 1
PRICE_MIN = 5.0
DECILE = 0.10
MICROCAP_PCTL = 0.20  # NYSE 20th percentile market cap breakpoint
WINSOR_LIMITS = (0.001, 0.001)  # 0.1% / 99.9% cross-sectional winsorization

# ============================================================
# STEP 1: Load Data & Clean Duplicates
# ============================================================
df = pd.read_csv("data.csv")
df.columns = df.columns.str.lower()
df["mthcaldt"] = pd.to_datetime(df["mthcaldt"])
df["ym"] = df["mthcaldt"].dt.to_period("M")

initial_len = len(df)

# 1. Drop exact duplicate rows
df = df.drop_duplicates()
exact_dups = initial_len - len(df)

# 2. Detect key conflicts [permno, ym]
conflict_mask = df.duplicated(subset=["permno", "ym"], keep=False)
conflict_count = conflict_mask.sum()

# 3. Drop all conflicting rows
df = df[~conflict_mask].reset_index(drop=True)

print(f"Initial rows: {initial_len}")
print(f"Exact duplicates removed: {exact_dups}")
print(f"Partial duplicate conflicts removed: {conflict_count}")
print(f"Final rows remaining: {len(df)}")
print("-" * 50)

# ============================================================
# STEP 2: Reindex to Complete Monthly Calendar (Fix Gap Months)
# ============================================================
def reindex_stock(g):
    permno_val = g.name
    g = g.set_index("ym").sort_index()
    full_months = pd.period_range(g.index.min(), g.index.max(), freq="M")
    g = g.reindex(full_months)
    g.index.name = "ym"

    g["permno"] = permno_val
    g["sharetype"] = g["sharetype"].ffill().bfill()
    g["primaryexch"] = g["primaryexch"].ffill().bfill()
    return g.reset_index()

df = df.groupby("permno", group_keys=False).apply(reindex_stock).reset_index(drop=True)

# ============================================================
# STEP 3: Eligibility Flag
# ============================================================
df["eligible"] = (
    (df["mthprc"].abs() >= PRICE_MIN) &
    (df["sharetype"] == "NS") &
    (df["primaryexch"].isin(["N", "A", "Q"]))
)

nyse_breakpoint = (
    df[(df["primaryexch"] == "N") & df["mthcap"].notna()]
    .groupby("ym")["mthcap"]
    .quantile(MICROCAP_PCTL)
    .rename("mktcap_p20")
)
df = df.merge(nyse_breakpoint, on="ym", how="left")
df["eligible"] = df["eligible"] & (df["mthcap"] >= df["mktcap_p20"])

# ============================================================
# STEP 3.5: Winsorize returns cross-sectionally, by month
#   Toggle: set APPLY_WINSORIZE = True/False below to switch between
#   the baseline (no winsorize) and the robustness-check version.
#   Run the whole script once with each setting and compare results_df.
# ============================================================
APPLY_WINSORIZE = True  # <- baseline: False. Robustness check run: True.

if APPLY_WINSORIZE:
    df["mthret"] = df.groupby("ym")["mthret"].transform(
        lambda x: winsorize(x, limits=WINSOR_LIMITS)
    )

# ============================================================
# STEP 4: Formation Period Return R_i,t(J)
# ============================================================
def compute_formation_returns(data, J):
    d = data.sort_values(["permno", "ym"]).copy()

    # check how many returns needed clipping before computing log returns --
    # a large count would suggest a data quality issue upstream, not just
    # a few genuine extreme observations
    n_clipped = (d["mthret"] < -0.9999).sum()
    print(f"[J={J}] returns clipped at -0.9999: {n_clipped}")

    clean_ret = d["mthret"].clip(lower=-0.9999)
    d["log_ret"] = np.log1p(clean_ret)

    d[f"R_J{J}"] = (
        d.groupby("permno")["log_ret"]
        .transform(lambda x: np.expm1(x.shift(SKIP + 1).rolling(J, min_periods=J).sum()))
    )
    return d

# ============================================================
# STEP 5: Rank into Deciles (Winners / Losers)
# ============================================================
def rank_winners_losers(data, J):
    col = f"R_J{J}"
    d = data.dropna(subset=[col]).copy()
    d = d[d["eligible"]].copy()

    d["lo"] = d.groupby("ym")[col].transform(lambda x: x.quantile(DECILE))
    d["hi"] = d.groupby("ym")[col].transform(lambda x: x.quantile(1 - DECILE))

    d["group"] = np.where(
        d[col] <= d["lo"], "Loser",
        np.where(d[col] >= d["hi"], "Winner", None)
    )

    return d[d["group"].isin(["Winner", "Loser"])].copy()

# ============================================================
# STEP 6: Build K-Month Cohort Contributions
# ============================================================
def build_cohort_contributions(ranked, raw, K):
    raw_ret = raw[["permno", "ym", "mthret"]].rename(columns={"mthret": "ret_t"})

    rows = []
    for s, g in ranked.groupby("ym"):
        win = g[g["group"] == "Winner"].dropna(subset=["mthcap"])
        los = g[g["group"] == "Loser"].dropna(subset=["mthcap"])
        if len(win) == 0 or len(los) == 0:
            continue

        w_win_initial = win.set_index("permno")["mthcap"]
        w_los_initial = los.set_index("permno")["mthcap"]

        for k in range(1, K + 1):
            t = s + k
            ret_t = raw_ret[raw_ret["ym"] == t].set_index("permno")["ret_t"]

            win_ret = ret_t.reindex(win["permno"]).dropna()
            los_ret = ret_t.reindex(los["permno"]).dropna()

            if len(win_ret) == 0 or len(los_ret) == 0:
                continue

            umd_ew = win_ret.mean() - los_ret.mean()

            w_win_t = w_win_initial.reindex(win_ret.index)
            w_win_t = w_win_t / w_win_t.sum()

            w_los_t = w_los_initial.reindex(los_ret.index)
            w_los_t = w_los_t / w_los_t.sum()

            umd_vw = (w_win_t * win_ret).sum() - (w_los_t * los_ret).sum()

            rows.append({"hold_month": t, "UMD_EW": umd_ew, "UMD_VW": umd_vw})

    return pd.DataFrame(rows)

# ============================================================
# STEP 7: Overlapping Portfolio Returns & Performance
# ============================================================
def overlapping_portfolio(contrib_df, col):
    return contrib_df.groupby("hold_month")[col].mean().sort_index()

def performance(returns):
    r = returns.dropna()
    n = len(r)
    if n == 0:
        return {"n_obs": 0, "mean_monthly": np.nan, "cum_return": np.nan, "t_stat": np.nan, "sharpe": np.nan}

    mu = r.mean()
    sigma = r.std(ddof=1)
    se = sigma / np.sqrt(n) if n > 0 else np.nan
    cr = np.prod(1 + r) - 1
    t_stat = mu / se if se and se != 0 else np.nan
    sharpe = (mu / sigma) * np.sqrt(12) if sigma and sigma != 0 else np.nan

    return {
        "n_obs": n,
        "mean_monthly": mu,
        "cum_return": cr,
        "t_stat": t_stat,
        "sharpe": sharpe
    }

# ============================================================
# MAIN EXECUTION
# ============================================================
results = []
monthly_series_all = []
for J in J_LIST:
    df_J = compute_formation_returns(df, J)
    ranked = rank_winners_losers(df_J, J)

    for K in K_LIST:
        contrib = build_cohort_contributions(ranked, df, K)

        for col in ["UMD_EW", "UMD_VW"]:
            port = overlapping_portfolio(contrib, col)
            metrics = performance(port)
            metrics.update({"J": J, "K": K, "weight": col.replace("UMD_", "")})
            results.append(metrics)

            # save the monthly return series itself -- needed for the
            # five-factor regression later (summary stats alone aren't enough)
            monthly_df = port.reset_index()
            monthly_df.columns = ["hold_month", "wml_return"]
            monthly_df["J"] = J
            monthly_df["K"] = K
            monthly_df["weight"] = col.replace("UMD_", "")
            monthly_series_all.append(monthly_df)

results_df = pd.DataFrame(results)[
    ["J", "K", "weight", "n_obs", "mean_monthly", "cum_return", "t_stat", "sharpe"]
]

print("\n--- Final Backtest Results ---")
print(results_df)

output_name = "wml_results_winsorized.csv" if APPLY_WINSORIZE else "wml_results_baseline.csv"
results_df.to_csv(output_name, index=False)
print(f"Saved to: {output_name}")

# save the monthly WML return series -- this is what you need for the
# five-factor regression
monthly_series_df = pd.concat(monthly_series_all, ignore_index=True)
monthly_name = "wml_monthly_winsorized.csv" if APPLY_WINSORIZE else "wml_monthly_baseline.csv"
monthly_series_df.to_csv(monthly_name, index=False)
print(f"Saved monthly series to: {monthly_name}")

Initial rows: 1048575
Exact duplicates removed: 13089
Partial duplicate conflicts removed: 1066
Final rows remaining: 1034420
--------------------------------------------------
[J=3] returns clipped at -0.9999: 0
[J=6] returns clipped at -0.9999: 0

--- Final Backtest Results ---
   J  K weight  n_obs  mean_monthly  cum_return    t_stat    sharpe
0  3  3     EW    128      0.006913    1.235019  2.223601  0.680836
1  3  3     VW    128      0.008340    1.513417  1.982318  0.606958
2  3  6     EW    128      0.005148    0.816264  1.887226  0.577842
3  3  6     VW    128      0.005224    0.732616  1.373690  0.420605
4  6  3     EW    125      0.005788    0.820481  1.460244  0.452440
5  6  3     VW    125      0.008124    1.142277  1.439147  0.445903
6  6  6     EW    125      0.003579    0.395491  0.946149  0.293154
7  6  6     VW    125      0.004326    0.377936  0.824960  0.255605
Saved to: wml_results_winsorized.csv
Saved monthly series to: wml_monthly_winsorized.csv


In [4]:
#merge 
import pandas as pd

baseline = pd.read_csv("wml_results_baseline.csv")
winsorized = pd.read_csv("wml_results_winsorized.csv")

compare = baseline.merge(
    winsorized, on=["J", "K", "weight"], suffixes=("_base", "_wins")
)

compare["t_stat_diff"] = compare["t_stat_wins"] - compare["t_stat_base"]
compare["sharpe_diff"] = compare["sharpe_wins"] - compare["sharpe_base"]

print(compare[["J", "K", "weight", "t_stat_base", "t_stat_wins", "t_stat_diff",
               "sharpe_base", "sharpe_wins", "sharpe_diff"]])

   J  K weight  t_stat_base  t_stat_wins  t_stat_diff  sharpe_base  \
0  3  3     EW     2.217329     2.223601     0.006271     0.678916   
1  3  3     VW     1.982041     1.982318     0.000277     0.606874   
2  3  6     EW     1.887242     1.887226    -0.000016     0.577847   
3  3  6     VW     1.375867     1.373690    -0.002176     0.421271   
4  6  3     EW     1.461650     1.460244    -0.001406     0.452876   
5  6  3     VW     1.444262     1.439147    -0.005115     0.447488   
6  6  6     EW     0.953362     0.946149    -0.007213     0.295389   
7  6  6     VW     0.831381     0.824960    -0.006421     0.257594   

   sharpe_wins  sharpe_diff  
0     0.680836     0.001920  
1     0.606958     0.000085  
2     0.577842    -0.000005  
3     0.420605    -0.000666  
4     0.452440    -0.000436  
5     0.445903    -0.001585  
6     0.293154    -0.002235  
7     0.255605    -0.001989  


In [5]:
df1=pd.read_csv("5Fac.csv",skiprows=3)
df1.head()

,Unnamed: 0,Mkt-RF,SMB,HML,RMW,CMA,RF
0,196307,-0.39,-0.48,-0.81,0.64,-1.15,0.27
1,196308,5.08,-0.80,1.70,0.40,-0.38,0.25
2,196309,-1.57,-0.43,0.00,-0.78,0.15,0.27
3,196310,2.54,-1.34,-0.04,2.79,-2.25,0.29
4,196311,-0.86,-0.85,1.73,-0.43,2.27,0.27


In [6]:
import pandas as pd
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant

# ============================================================
# STEP 1: Load and clean the 5-factor file safely
# ============================================================
# Safely locate the header line containing 'Mkt-RF' using line-by-line reading 
# to prevent pandas ParserError caused by irregular description text at the top.
header_idx = None
with open("5Fac.csv", "r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        if "Mkt-RF" in line:
            header_idx = idx
            break

if header_idx is None:
    raise ValueError("Header containing 'Mkt-RF' was not found in 5Fac.csv!")

# Read data skipping all non-data header lines
factors = pd.read_csv("5Fac.csv", skiprows=header_idx)
factors = factors.rename(columns={factors.columns[0]: "yyyymm"})

# Clean yyyymm column and filter for monthly observations (>= 6-digit integers)
factors["yyyymm"] = pd.to_numeric(factors["yyyymm"], errors="coerce")
factors = factors.dropna(subset=["yyyymm"])
factors = factors[factors["yyyymm"] >= 100000].copy()
factors["yyyymm"] = factors["yyyymm"].astype(int)

# Convert YYYYMM integer to monthly Period format matching hold_month
factors["hold_month"] = pd.to_datetime(factors["yyyymm"], format="%Y%m").dt.to_period("M")

# Convert factor returns from percentages to decimal values
factor_cols = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]
factors[factor_cols] = factors[factor_cols].astype(float) / 100

factors = factors[["hold_month"] + factor_cols]

# ============================================================
# STEP 2: Load WML monthly returns (baseline, no winsorize)
# ============================================================
wml = pd.read_csv("wml_monthly_baseline.csv")
wml["hold_month"] = pd.PeriodIndex(wml["hold_month"], freq="M")

# ============================================================
# STEP 3: Merge -- keep only months that exist in BOTH files
# ============================================================
merged = wml.merge(factors, on="hold_month", how="inner")
print("Months matched:", merged["hold_month"].nunique())

# ============================================================
# STEP 4: Run 5-factor regression with Newey-West (HAC) standard errors
# ============================================================
results = []
factor_x_cols = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"]

for (J, K, weight), g in merged.groupby(["J", "K", "weight"]):
    y = g["wml_return"]
    X = g[factor_x_cols]
    
    # Add intercept column (Alpha) explicitly
    X = add_constant(X)

    # Dynamic lag selection: K - 1 lags for overlapping returns (minimum 1 lag)
    max_lags = max(1, K - 1)

    # Fit OLS model with Newey-West (HAC) standard errors adjustment
    model = OLS(y, X)
    fit_res = model.fit(cov_type="HAC", cov_kwds={"maxlags": max_lags})

    # Store regression outputs
    results.append({
        "J": J,
        "K": K,
        "weight": weight,
        "alpha": fit_res.params["const"],
        "alpha_t_stat_NW": fit_res.tvalues["const"],  # Newey-West adjusted t-statistic
        "beta_mkt": fit_res.params["Mkt-RF"],
        "beta_smb": fit_res.params["SMB"],
        "beta_hml": fit_res.params["HML"],
        "beta_rmw": fit_res.params["RMW"],
        "beta_cma": fit_res.params["CMA"],
        "r_squared": fit_res.rsquared,
        "n_obs": int(fit_res.nobs)
    })

results_df = pd.DataFrame(results)
print("\n--- Regression Results (Newey-West Adjusted) ---")
print(results_df)

# Save output to CSV
results_df.to_csv("five_factor_regression_results.csv", index=False)

Months matched: 128

--- Regression Results (Newey-West Adjusted) ---
   J  K weight     alpha  alpha_t_stat_NW  beta_mkt  beta_smb  beta_hml  \
0  3  3     EW  0.008035         2.803893 -0.053545 -0.163238 -0.266171   
1  3  3     VW  0.009133         2.305774 -0.068795 -0.264391 -0.389258   
2  3  6     EW  0.006259         2.763540 -0.110853 -0.264655 -0.190141   
3  3  6     VW  0.005752         2.042837 -0.083968 -0.400039 -0.360274   
4  6  3     EW  0.007350         2.379718 -0.152843 -0.401032 -0.263643   
5  6  3     VW  0.009227         2.115234 -0.146861 -0.771164 -0.419901   
6  6  6     EW  0.005280         1.826130 -0.175324 -0.473659 -0.224212   
7  6  6     VW  0.004911         1.203479 -0.105829 -0.775489 -0.406732   

   beta_rmw  beta_cma  r_squared  n_obs  
0 -0.266761  0.541486   0.148063    128  
1 -0.258757  0.352761   0.135754    128  
2 -0.148728  0.394564   0.216409    128  
3 -0.193114  0.381934   0.213778    128  
4 -0.231980  0.637427   0.237287    125  
5 